In [2]:
import os

In [22]:
folder = "/home/simon/Desktop/yolo-training/SportsInnovation Game1/job_2916-2026_03_10_23/obj_train_data"

all_files = os.listdir(folder)
imgs = [f for f in all_files if f.endswith('.png')]
txts = [f for f in all_files if f.endswith('.txt')]

names_file = "/home/simon/Desktop/yolo-training/CVAT Log/obj.names"

with open(names_file, 'r') as f:
    name_conversion = {name.removesuffix("\n"): i for i, name in enumerate(f.readlines())}
    new_ids = {"ball": 0,
               "robot": 1,
               "penalty cross": 2}

# First value of tuple is replaced by second value
replaces = [("SPL Ball", "ball"),
            ("FIFA 26 Ball", "ball"),
            # ("Nao", "robot"),
            ("K1", "robot"),
            ("PenaltyMark", "penalty cross")]
replaces_int = [(name_conversion[old], new_ids[new]) for old, new in replaces]



In [23]:
for t in txts:
    with open(os.path.join(folder, t), 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.split()
        class_id = int(parts[0])
        if class_id in [0, 1, 3, 5]:
            for old_id, new_id in replaces_int:
                if class_id == old_id:
                    parts[0] = str(new_id)
                    # print(f"Replaced class ID {old_id} with {new_id} in file {t}")
                    break
            new_lines.append(" ".join(parts))

    with open(os.path.join(folder, t), 'w') as f:
        f.write("\n".join(new_lines))

# Fill missing empty labels

In [44]:
imgs = [f for f in os.listdir(folder) if f.endswith('.png')]
txts = [f for f in os.listdir(folder) if f.endswith('.txt')]

count = 0
for img in imgs:
    txt_file = img.replace('.png', '.txt')
    if txt_file not in txts:
        with open(os.path.join(folder, txt_file), 'w') as f:
            f.write("")
            count+=1
print(f"Created {count} empty label files.")

Created 0 empty label files.


# Replace IDs (back from 3cls to 11cls)

In [45]:
# path = "/home/simon/Downloads/job_2915-2026_03_10_11_40_52-yolo 1.1/obj_train_data"

# Fist gets replaced by second value
replaces = {0: 0,
            1: 3,
            2: 5}

label_files = [f for f in all_files if f.endswith('.txt')] #[p for p in os.listdir(path) if p.endswith(".txt")]
for label_file in label_files:
    with open(os.path.join(folder, label_file), 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.split()
        parts[0] = str(replaces[int(parts[0])])
        new_lines.append(" ".join(parts))

    with open(os.path.join(folder, label_file), 'w') as f:
        f.write("\n".join(new_lines))

# Convert Robert format to YOLO format

In [60]:
from pathlib import Path

path = "/home/simon/Desktop/yolo-training/RobertData/labels"
out_path = "/home/simon/Desktop/yolo-training/RobertData/labels_yolo"
Path(out_path).mkdir(parents=True, exist_ok=True)
label_files = [f for f in os.listdir(path) if f.endswith('.txt')]

In [63]:
def parse_bbox_line(line):
    line_split = line.split(":")
    cls, coords = line_split[0].strip(), line_split[1].strip()
    class_id = name_to_id[cls]

    if len(coords.split(" ")) != 4:
        print(f"Skipping line in file {label_file} due to incorrect format: {line}")
        return ""
    new_line = f"{class_id} {coords}"
    return new_line

def parse_ppoint_line(line):
    if len(line.split(" ")) != 4:
        print(f"Skipping line in file {label_file} due to incorrect format: {line}")
        return ""
    new_line = f"2 {line.strip()}"
    return new_line

In [64]:
name_to_id = {
    "Trionda Ball 2026(Clone)": 0,
    "K1(Clone)": 1
}

headers = ["BoundingBoxes", "GoalPosts", "CenterCircle", "PenaltyPoints", "Lines"]  # Add more headers if needed
currentSection = ""

for label_file in label_files:
    with open(os.path.join(path, label_file), 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        if line.split(":")[0] in headers:
            currentSection = line.split(":")[0]
            continue

        match currentSection:
            case "BoundingBoxes":
                nl = parse_bbox_line(line)
                if nl != "":
                    new_lines.append(nl)
            case "GoalPosts":
                pass
            case "CenterCircle":
                pass
            case "PenaltyPoints":
                nl = parse_ppoint_line(line)
                if nl != "":
                    new_lines.append(nl)
            case "Lines":
                pass
            case _:
                print(f"Unknown section {currentSection} in file {label_file}")
                continue

    with open(os.path.join(out_path, label_file), 'w') as f:
        f.write("\n".join(new_lines))